# UKB Pathogenic Mutation Summary

This notebook processes all `*.raw` files in:

`analysis/06-UKB_analyisis/00-Data-prepration/3.Genotype/3.Other/gene_extraction_results`

## What it does
- Reads all gene-specific `.raw` files.
- Exports the first 12 lines of each file into one combined text file.
- Builds a mutation summary table where each row is one mutation.

## Output columns
- `gene`: parsed from filename (`UKB_<gene>_pathogenic_corrected.raw`)
- `mutation`: mutation column name from `.raw`
- `carried_count`: number of participants with genotype value `> 0`

## Generated files
- `all_raw_heads_1_12.txt`
- `mutation_summary_table.tsv`

Run cells from top to bottom.

In [ ]:
# ---- Setup ----
input_dir <- "./gene_extraction_results"
head_lines <- 12L

# Resolve folder robustly for notebook execution from different working directories.
if (!dir.exists(input_dir)) {
  input_dir <- file.path(getwd(), input_dir)
}
input_dir <- normalizePath(input_dir, winslash = "/", mustWork = TRUE)

heads_output <- file.path(input_dir, "all_raw_heads_1_12.txt")
summary_output <- file.path(input_dir, "mutation_summary_table.tsv")

raw_files <- sort(list.files(input_dir, pattern = "\\.raw$", full.names = TRUE))
stopifnot(length(raw_files) > 0)

extract_gene <- function(file_name) {
  m <- regexec("^UKB_(.+?)_pathogenic_corrected\\.raw$", file_name)
  hit <- regmatches(file_name, m)[[1]]
  if (length(hit) >= 2) hit[2] else tools::file_path_sans_ext(file_name)
}

cat("Raw files found:", length(raw_files), "\n")

In [ ]:
# ---- 1) Extract first N lines from each .raw ----
# Build content in memory and write once (avoids notebook connection-state issues).
all_lines <- character(0)

for (raw_file in raw_files) {
  file_name <- basename(raw_file)
  all_lines <- c(all_lines, sprintf("@%s (1-%d)", file_name, head_lines))
  lines <- readLines(raw_file, n = head_lines, warn = FALSE, encoding = "UTF-8")
  if (length(lines) > 0) {
    all_lines <- c(all_lines, lines)
  }
  all_lines <- c(all_lines, "")
}

writeLines(all_lines, con = heads_output, useBytes = TRUE)

In [ ]:
# ---- 2) Build mutation summary table ----
summary_list <- vector("list", length(raw_files))

for (i in seq_along(raw_files)) {
  raw_file <- raw_files[i]
  gene <- extract_gene(basename(raw_file))

  header <- readLines(raw_file, n = 1, warn = FALSE, encoding = "UTF-8")
  if (length(header) == 0) {
    summary_list[[i]] <- data.frame(
      gene = character(0),
      mutation = character(0),
      carried_count = integer(0),
      stringsAsFactors = FALSE
    )
    next
  }

  cols <- strsplit(header, "\t", fixed = TRUE)[[1]]
  if (length(cols) <= 6) {
    summary_list[[i]] <- data.frame(
      gene = character(0),
      mutation = character(0),
      carried_count = integer(0),
      stringsAsFactors = FALSE
    )
    next
  }

  mutation_cols <- cols[7:length(cols)]

  df <- utils::read.table(
    raw_file,
    sep = "\t",
    header = TRUE,
    quote = "",
    comment.char = "",
    fill = TRUE,
    check.names = FALSE,
    stringsAsFactors = FALSE
  )

  geno <- df[, 7:ncol(df), drop = FALSE]

  carried_counts <- vapply(
    geno,
    FUN = function(x) {
      suppressWarnings(x_num <- as.numeric(x))
      sum(!is.na(x_num) & x_num > 0)
    },
    FUN.VALUE = integer(1)
  )

  summary_list[[i]] <- data.frame(
    gene = rep(gene, length(mutation_cols)),
    mutation = mutation_cols,
    carried_count = as.integer(carried_counts),
    stringsAsFactors = FALSE
  )
}

summary_df <- do.call(rbind, summary_list)
utils::write.table(summary_df, summary_output, sep = "\t", row.names = FALSE, quote = FALSE)

cat("Rows:", nrow(summary_df), "\n")

In [ ]:
# save the result to csv file
write.csv(summary_df, file = "mutation_summary_table.csv", row.names = FALSE)
